In [ ]:
repository_filter: list[str] = []
data_file_npm: str = (
    "../samples/v2/org.openrewrite.nodejs.table.VulnerabilityReport.csv"
)
data_file_pypi: str = (
    "../samples/v2/org.openrewrite.python.dependencies.table.VulnerabilityReport.csv"
)
# No NuGet sample ships with this repository, so the default reads as an empty table
data_file_nuget: str = ""
max_vulnerabilities_per_repository: int = 50

In [ ]:
from moderne_visualizations_misc.reusable.data_loader import (
    read_data_table,
    read_optional_csv,
)
from moderne_visualizations_misc.reusable import quality_utils as qu
from code_data_science import tree_data_grid
import pandas as pd
import re
import warnings

warnings.simplefilter("ignore")

df = read_data_table(
    "../samples/v2/io.moderne.devcenter.table.UpgradesAndMigrations.csv"
)
df = qu.filter_repos(df, repository_filter)
# Rows without a card or category can't be placed in the tree
df = df.dropna(subset=["card", "value"])

# Each ecosystem card reads one report; Java cards write none, so they stop at the repository
reports = {
    "npm": read_optional_csv(data_file_npm),
    "pypi": read_optional_csv(data_file_pypi),
    "nuget": read_optional_csv(data_file_nuget),
}

# Match repositories on text keys, so an empty branch still matches its report rows
REPOSITORY_KEY = ["repositoryOrigin", "repositoryPath", "repositoryBranch"]
for table in [df, *reports.values()]:
    for column in REPOSITORY_KEY:
        if column in table.columns:
            table[column] = table[column].fillna("").astype(str)

In [ ]:
SEVERITY_ORDER = {"CRITICAL": 0, "HIGH": 1, "MODERATE": 2, "MEDIUM": 2, "LOW": 3}
# Severity card categories; unfixable vulnerabilities fall in the last one
SEVERITY_CATEGORIES = {
    "Critical",
    "High",
    "Moderate",
    "Low",
    "No fixable vulnerabilities",
}


def text(value) -> str:
    return "" if pd.isna(value) else str(value)


def ecosystem_of(card: str) -> str | None:
    """Default card names start with the ecosystem label, such as `npm vulnerabilities by Severity`."""
    first = card.split(" ", 1)[0].lower()
    return first if first in reports else None


def organizations(row: pd.Series, org_columns: list[str]) -> str:
    """Platform data tables list the repository's organization in `org1` and its parents in `org2`, `org3`, ..."""
    return " / ".join(text(row[c]) for c in reversed(org_columns) if text(row[c]))


def by_repository(report: pd.DataFrame) -> dict:
    """Each repository's distinct vulnerabilities, worst severity first, grouped once per report."""
    if report.empty:
        return {}
    # Reports have a row per project, so list each vulnerability in a package version once
    report = report.drop_duplicates(
        subset=REPOSITORY_KEY + ["cve", "packageName", "version"]
    )
    rank = report["severity"].map(lambda s: SEVERITY_ORDER.get(text(s).upper(), 4))
    report = report.assign(_rank=rank).sort_values(["_rank", "packageName", "cve"])
    return {key: rows for key, rows in report.groupby(REPOSITORY_KEY, sort=False)}


def listed_under(vulnerabilities: pd.DataFrame, category: str) -> pd.DataFrame:
    """A repository sits only in its worst category, so it lists every vulnerability.

    Under `No fixable vulnerabilities`, it lists only the unfixable ones.
    """
    if category == "No fixable vulnerabilities":
        fixed = vulnerabilities["fixedVersion"].map(text)
        return vulnerabilities[fixed == ""]
    return vulnerabilities


def repository_labels(rows: pd.DataFrame) -> list[str]:
    """Label by path, adding the branch, and then the origin, only where the path alone repeats."""
    same_path = rows["repositoryPath"].duplicated(keep=False)
    same_branch = rows.duplicated(
        subset=["repositoryPath", "repositoryBranch"], keep=False
    )
    labels = []
    for (_, row), path_repeats, branch_repeats in zip(
        rows.iterrows(), same_path, same_branch
    ):
        label = text(row["repositoryPath"])
        if branch_repeats:
            label += (
                f" ({text(row['repositoryOrigin'])}, {text(row['repositoryBranch'])})"
            )
        elif path_repeats:
            label += f" ({text(row['repositoryBranch'])})"
        labels.append(label)
    return labels


def repository_count(rows: pd.DataFrame) -> int:
    return len(rows.drop_duplicates(subset=REPOSITORY_KEY))


if df.empty:
    tree_data_grid.display(
        [{"path": ["No DevCenter card results for the selected filters"]}],
        "Card",
        [],
    )
else:
    org_columns = sorted(
        (c for c in df.columns if re.fullmatch(r"org\d+", c)), key=lambda c: int(c[3:])
    )
    df["ordinal"] = pd.to_numeric(df["ordinal"], errors="coerce")
    df = df.assign(_card=df["card"].str.lower()).sort_values(
        ["_card", "ordinal", "repositoryPath"]
    )
    vulnerabilities = {
        ecosystem: by_repository(report) for ecosystem, report in reports.items()
    }

    tree = []
    for card, card_rows in df.groupby("card", sort=False):
        tree.append({"path": [card], "repositories": repository_count(card_rows)})
        ecosystem = ecosystem_of(card)
        # Only Severity cards list vulnerabilities; each repository is in one category per card, so no list repeats
        lists_vulnerabilities = (
            ecosystem is not None and set(card_rows["value"]) <= SEVERITY_CATEGORIES
        )
        for category, category_rows in card_rows.groupby("value", sort=False):
            tree.append(
                {
                    "path": [card, category],
                    "repositories": repository_count(category_rows),
                }
            )
            for label, (_, repository) in zip(
                repository_labels(category_rows), category_rows.iterrows()
            ):
                repository_path = [card, category, label]
                tree.append(
                    {
                        "path": repository_path,
                        "organizations": organizations(repository, org_columns),
                        "branch": text(repository["repositoryBranch"]),
                        "details": text(repository["currentMinimumVersion"]),
                    }
                )
                if ecosystem is None or not lists_vulnerabilities:
                    continue
                key = tuple(repository[REPOSITORY_KEY])
                found = vulnerabilities[ecosystem].get(key, pd.DataFrame())
                if found.empty:
                    continue
                listed = listed_under(found, category)
                # Large organizations can have hundreds of vulnerabilities per repository
                if 0 < max_vulnerabilities_per_repository < len(listed):
                    hidden = len(listed) - max_vulnerabilities_per_repository
                    listed = listed.head(max_vulnerabilities_per_repository)
                else:
                    hidden = 0
                for _, vulnerability in listed.iterrows():
                    fixed = text(vulnerability.get("fixedVersion"))
                    tree.append(
                        {
                            "path": repository_path
                            + [
                                f"{text(vulnerability['cve'])} "
                                f"({text(vulnerability['packageName'])} {text(vulnerability['version'])})"
                            ],
                            "severity": text(vulnerability["severity"]).capitalize(),
                            "details": f"Fixed in {fixed}"
                            if fixed
                            else "No fix available",
                        }
                    )
                if hidden:
                    tree.append(
                        {
                            "path": repository_path
                            + [
                                f"{hidden} more vulnerabilit{'y' if hidden == 1 else 'ies'}"
                            ],
                            "details": "Raise the vulnerabilities per repository limit to list them",
                        }
                    )

    tree_data_grid.display(
        tree,
        "Card",
        [
            {"field": "repositories", "headerName": "Repositories", "minWidth": 110},
            {"field": "organizations", "headerName": "Organizations", "minWidth": 200},
            {"field": "branch", "headerName": "Branch", "minWidth": 100},
            {"field": "severity", "headerName": "Severity", "minWidth": 100},
            {"field": "details", "headerName": "Details", "minWidth": 260},
        ],
    )